# Chapter 2 — Colour–magnitude diagrams, isochrones and distance

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/02_cmd_isochrones.ipynb)

**Goal.** The stars of a satellite share an age, a metallicity and a distance, so they occupy a narrow sequence in
the colour–magnitude diagram (CMD). Here we isolate that sequence for Boötes I by subtracting the field, compare
it with theoretical isochrones, and measure the distance with a likelihood.

**You will learn**
1. to build **Hess diagrams** (binned CMDs) and subtract a background scaled by area;
2. what an **isochrone** is, and how the distance modulus $\mu = 5 \log_{10}(D / 10\,{\rm pc})$ shifts it;
3. to write a **mixture likelihood** of satellite and field stars in CMD space and profile it over distance.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')
ref = lvdb.loc['bootes_1']                                          # LVDB v1.1.1: centre [deg], mu [mag], r_h [arcmin]
cat = load('bootes1_sdss')                                          # SDSS DR9 point sources, g < 23, within 2 deg
# dereddened magnitudes and colour [mag], as plain numpy arrays
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
r0 = (cat.psfmag_r - cat.extinction_r).to_numpy()
c0 = g0 - r0
R = np.hypot(*tangent(cat.ra, cat.dec, ref.ra, ref.dec)) * 60      # arcmin
print(f'Boötes I (LVDB v1.1.1): mu = {ref.distance_modulus:.2f} ± {ref.distance_modulus_em:.2f}, r_h = {ref.rhalf:.2f}′')

## 2.1 Hess diagrams and background subtraction

Take the stars inside a radius $R_{\rm on}$ and those in an outer annulus. The annulus contains field stars only;
scaled by the ratio of areas $\alpha = A_{\rm on}/A_{\rm off}$, its Hess diagram predicts the field contribution
to the inner region: $\Delta H = H_{\rm on} - \alpha H_{\rm off}$. Each bin of $\Delta H$ has variance
$H_{\rm on} + \alpha^2 H_{\rm off}$ (Poisson), so the subtracted diagram is noisy and can be negative.

In [ ]:
# On-region: inside 1.5 r_h (15′); for a round exponential or Plummer profile this circle holds ~70% of the
# satellite's stars. Off-region: the 60′-120′ annulus (6-12 r_h), field stars only.
on = R < 1.5 * ref.rhalf
off = (R > 60) & (R < 120)
alpha = (1.5 * ref.rhalf)**2 / (120**2 - 60**2)                     # area ratio A_on / A_off
# Hess-diagram bins: 0.08 mag in (g-r)_0 from -0.4 to 1.36 (22 bins) and 0.2 mag in g_0 from 16 to 22.8 (34 bins);
# the ~1400 on-region stars then give ~2 stars per bin on average.
cb, mb = np.arange(-0.4, 1.41, 0.08), np.arange(16, 22.81, 0.2)
H_on = np.histogram2d(c0[on], g0[on], [cb, mb])[0]                  # shape (22, 34): colour x magnitude
H_off = np.histogram2d(c0[off], g0[off], [cb, mb])[0]
dH = H_on - alpha * H_off                                           # background-subtracted Hess diagram
# Poisson error of the total excess: sqrt(N_on + alpha^2 N_off)
print(f'stars: on {on.sum()}, off {off.sum()}, area ratio alpha = {alpha:.4f}; '
      f'excess {dH.sum():.0f} ± {np.sqrt(H_on.sum() + alpha**2 * H_off.sum()):.0f}')

iso = load('isochrones_sdss')                                       # PARSEC 1.2S, absolute SDSS ugriz magnitudes
print('isochrone grid: log age', np.unique(iso.logage).round(2), '; [Fe/H]', np.unique(iso.feh).round(1))

An **isochrone** gives the magnitudes of stars of one age and metallicity as a function of their initial mass.
The file holds PARSEC 1.2S isochrones (Bressan et al. 2012) in SDSS bands. Boötes I is old and very metal-poor
([Fe/H] ≈ −2.5); the grid stops at [Fe/H] = −2.1, which we use with the oldest age, 12.6 Gyr (log age = 10.1). The
`stage` column labels evolutionary phases (1: main sequence, 2: subgiant, 3: red-giant branch, 4: horizontal
branch, ...). We drop the late phases beyond the asymptotic giant branch and shift the isochrone by $\mu$.

In [ ]:
# The oldest, most metal-poor isochrone of the grid (12.6 Gyr, [Fe/H] = -2.1), phases up to stage 7 (the early AGB
# in the PARSEC labels), sorted by initial mass so that it is drawn along the evolutionary sequence.
# np.isclose because logage and feh are stored as floats.
I = iso[(np.isclose(iso.logage, 10.1)) & (np.isclose(iso.feh, -2.1)) & (iso.stage <= 7)].sort_values('m_ini')
ic, ig = (I.g - I.r).to_numpy(), I.g.to_numpy()                     # colour and absolute g magnitude (186 points)
fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
ext = [cb[0], cb[-1], mb[-1], mb[0]]                                # imshow extent: faint magnitudes at the bottom
# left: on-region counts; middle: scaled field; right: difference on a diverging scale fixed at +-4 stars per bin
for a, h, t in [(ax[0], H_on, f'R < {1.5 * ref.rhalf:.0f}′'), (ax[1], alpha * H_off, 'annulus 60′–120′, × α'),
                (ax[2], dH, 'difference')]:
    im = a.imshow(h.T, extent=ext, aspect='auto', cmap='Greys' if a is not ax[2] else 'RdBu_r',
                  vmin=None if a is not ax[2] else -4, vmax=None if a is not ax[2] else 4)
    a.set(xlabel='$(g-r)_0$', title=t)
    plt.colorbar(im, ax=a)
ax[0].set(ylabel='$g_0$', ylim=(mb[-1], mb[0]))
# the isochrone shifted to apparent magnitude by the LVDB distance modulus (add values to the tuple to compare)
for mu in (ref.distance_modulus,):
    ax[2].plot(ic, ig + mu, 'k-', lw=1, label=f'PARSEC 12.6 Gyr, [Fe/H] = −2.1, μ = {mu:.2f}')
ax[2].legend(loc='lower left', fontsize=8)
plt.tight_layout()

**Figure 2.1.** Hess diagrams of SDSS DR9 point sources around Boötes I (`bootes1_sdss`, extinction-corrected):
star counts in bins of 0.08 mag in $(g-r)_0$ and 0.2 mag in $g_0$. Left: the 1387 stars inside 1.5 $r_h$ = 15′
(grey scale, stars per bin). Middle: the 43 316 stars of the 60′–120′ annulus multiplied by the area ratio
α = 0.0207, i.e. the field expected inside 15′. Right: their difference, on − α·off (red positive, blue negative;
colour scale clipped at ±4 stars per bin), an excess of 415 ± 33 stars in total, with the PARSEC isochrone of
12.6 Gyr and [Fe/H] = −2.1 shifted to the LVDB distance modulus μ = 19.11 (black line). The straight black segment
from the red-giant-branch tip at $g_0 \approx 17$ to the blue end of the horizontal branch only joins consecutive
isochrone points; the isochrone's red lower main sequence, far below the plotted magnitudes, stretches the colour
axis of this panel to ≈ 1.9. The excess (red) follows the isochrone at the published distance: a red-giant branch
from $g_0 \approx 18$ to the limit, and a blue horizontal branch at $g_0 \approx 19.6$. The field is dominated by
the "blue edge" of halo main-sequence turn-off stars at $(g-r)_0 \approx 0.2$–0.4 and by disc M dwarfs at
$(g-r)_0 > 1.2$.

## 2.2 A likelihood for the distance

We treat each star inside the on-region as drawn from a mixture of the satellite population and the field:

$$ \mathcal{L}(\mu, f) = \prod_i \big[ f\, s(c_i, m_i \mid \mu) + (1 - f)\, b(c_i, m_i) \big], $$

where $c = (g-r)_0$, $m = g_0$, $f$ is the satellite fraction, $s$ is the probability density of satellite stars
in the CMD, and $b$ that of field stars. Both are normalised over the same CMD region.

- $b$ is estimated empirically from the annulus (a smoothed histogram).
- $s$ is the isochrone shifted by $\mu$, with each point weighted by the number of stars per unit mass (from the
  initial mass function: `int_imf` is its cumulative integral), and blurred by the photometric errors at its
  magnitude (measured from the data) plus a small intrinsic width.

For each $\mu$ we maximise over $f$ — the **profile likelihood** — and plot $2\Delta \ln \mathcal L$, which near its
minimum behaves like a $\chi^2$ with one degree of freedom (1σ at 1).

In [ ]:
from scipy.ndimage import gaussian_filter
from scipy.optimize import minimize_scalar

# CMD region of the likelihood: -0.4 < (g-r)_0 < 1.4 and 16 < g_0 < 22.5 (about the 10-sigma depth, Chapter 0)
box = (c0 > -0.4) & (c0 < 1.4) & (g0 > 16) & (g0 < 22.5)
# photometric errors as a function of magnitude: median error in magnitude bins
# 0.25-mag bins from 14 to 23 in observed PSF magnitude (every bin holds >= 150 stars here); em = bin centres
eb = np.arange(14, 23.01, 0.25)
em = 0.5 * (eb[1:] + eb[:-1])
med_err = {b: np.array([np.nanmedian(cat[f'psfmagerr_{b}'][(cat[f'psfmag_{b}'] >= lo) & (cat[f'psfmag_{b}'] < hi)])
                        for lo, hi in zip(eb[:-1], eb[1:])]) for b in 'gr'}
# sig(b, m): median error [mag] in band b at magnitude m, interpolated (constant beyond the end bins). It is evaluated
# at dereddened isochrone magnitudes, ~0.08 mag (A_g) brighter than the observed ones: a small shift.
# nan=0.3 would give a large error to an empty bin (there are none here).
sig = lambda b, m: np.interp(m, em, np.nan_to_num(med_err[b], nan=0.3))

# Sample the isochrone uniformly along its length in the CMD (uniform in mass would leave the fast phases,
# e.g. the red-giant branch, almost empty), and weight each point by the number of stars it represents:
# the increment of the cumulative IMF, int_imf.
arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(ic), np.diff(ig)))])   # path length in the (g-r, g) plane [mag]
# 6000 points along a 23.5-mag path: steps of 0.004 mag, far finer than the photometric errors (>= 0.015 mag)
s_fine = np.linspace(0, arc[-1], 6000)
pg, pr = np.interp(s_fine, arc, I.g), np.interp(s_fine, arc, I.r)  # absolute g, r of the sample points
w_imf = np.gradient(np.interp(s_fine, arc, I.int_imf))             # stars per sample point (arbitrary normalisation)
# CMD grid for normalisation
# 90 colour bins of 0.02 mag x 130 magnitude bins of 0.05 mag covering the box; GC, GM: bin centres, shape (90, 130)
gc = np.linspace(-0.4, 1.4, 91); gm = np.linspace(16, 22.5, 131)
GC, GM = np.meshgrid(0.5 * (gc[1:] + gc[:-1]), 0.5 * (gm[1:] + gm[:-1]), indexing='ij')
cell = (gc[1] - gc[0]) * (gm[1] - gm[0])                           # area of a grid cell [mag^2]


def sat_density(c, m, mu, intrinsic=0.03, depth=0.1):
    """Satellite CMD density at (c, m) for distance modulus mu, normalised over the CMD box.
    Each isochrone point is blurred by the photometric errors plus an intrinsic width in colour and in magnitude
    (the latter stands for the spread in distance and in horizontal-branch luminosity).

    c, m: colour (g-r)_0 and magnitude g_0 [mag], arrays of any shape; intrinsic, depth: the extra widths in colour
    and in magnitude [mag]. Returns a flat array of densities [mag^-2] whose integral over the box is 1.
    """
    g, r = pg + mu, pr + mu                                        # apparent magnitudes of the isochrone points
    # keep points from 15 to 23.5, a margin around the box (16-22.5), so that points just outside blur into it
    keep = (g < 23.5) & (g > 15)
    g, r, w = g[keep], r[keep], w_imf[keep]
    # Gaussian widths per point, added in quadrature: colour error from independent g and r errors plus the intrinsic
    # width; magnitude error plus the depth term
    sc = np.hypot(np.hypot(sig('g', g), sig('r', r)), intrinsic)
    sm = np.hypot(sig('g', g), depth)

    def dens(cc, mm):
        """IMF-weighted sum of 2-D Gaussians (one per isochrone point) at the points (cc, mm)."""
        out = np.zeros(cc.size)
        for j in range(0, g.size, 500):                       # chunks keep memory small
            k = slice(j, j + 500)
            # (number of evaluation points, 500) array of Gaussian values, summed over the isochrone points
            out += (w[k] * np.exp(-0.5 * ((cc.ravel()[:, None] - (g[k] - r[k])) / sc[k])**2 - 0.5 * ((mm.ravel()[:, None] - g[k]) / sm[k])**2)
                    / (2 * np.pi * sc[k] * sm[k])).sum(1)
        return out
    norm = dens(GC, GM).sum() * cell                               # integral over the box, on the grid
    return dens(np.asarray(c), np.asarray(m)) / norm


# field density from the annulus: a smoothed, normalised histogram on the same grid
# Smoothing: Gaussian of 1 grid cell (0.02 mag in colour, 0.05 mag in g). The floor of 1e-3 stars per cell keeps
# the field density positive everywhere, so that ln L stays finite for a star in an empty cell.
Hb = gaussian_filter(np.histogram2d(c0[off & box], g0[off & box], [gc, gm])[0], 1.0) + 1e-3
Hb /= Hb.sum() * cell                                              # density [mag^-2], unit integral over the box
# b(c, m): look up the grid cell of each star (indices clipped so that edge values fall in the edge cells)
bg_density = lambda c, m: Hb[np.clip(np.digitize(c, gc) - 1, 0, len(gc) - 2), np.clip(np.digitize(m, gm) - 1, 0, len(gm) - 2)]

sel = on & box                                                     # the stars entering the likelihood
b_i = bg_density(c0[sel], g0[sel])                                 # field density at each star: independent of mu
def profile(mu):
    """-ln L maximised over the satellite fraction f, at distance modulus mu."""
    s_i = sat_density(c0[sel], g0[sel], mu)
    # bounded 1-d minimisation of -ln L over 0 <= f <= 1; .fun is the minimum value
    return minimize_scalar(lambda f: -np.sum(np.log(f * s_i + (1 - f) * b_i)), bounds=(0, 1), method='bounded').fun


# coarse: 0.1-mag steps over 17.5-21.0 (32-158 kpc); fine: 0.01-mag steps within +-0.3 mag of the coarse minimum
coarse = np.arange(17.5, 21.01, 0.1)                       # a coarse scan, then a fine one around its minimum
pc = np.array([profile(mu) for mu in coarse])
fine = np.arange(coarse[np.argmin(pc)] - 0.3, coarse[np.argmin(pc)] + 0.3, 0.01)
pf = np.array([profile(mu) for mu in fine])
# merge both scans, sort by mu, and convert -ln L to 2 Delta ln L relative to the best value
mus = np.concatenate([coarse, fine]); order = np.argsort(mus)
mus, prof = mus[order], np.concatenate([pc, pf])[order]
prof = 2 * (prof - prof.min())
mu_hat = mus[np.argmin(prof)]
# 68% interval: grid values with 2 Delta ln L < 1 (chi^2 with 1 degree of freedom); resolution 0.01 mag
in1 = mus[prof < 1]
print(f'mu = {mu_hat:.2f}  (1-sigma range {in1.min():.2f}–{in1.max():.2f}); LVDB: {ref.distance_modulus:.2f} ± {ref.distance_modulus_em:.2f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
# left: the profile likelihood, the 1-sigma level (dotted) and the LVDB value (dashed)
ax[0].plot(mus, prof)
ax[0].axhline(1, c='k', lw=0.8, ls=':')
ax[0].axvline(ref.distance_modulus, c='C3', ls='--', label='LVDB')
ax[0].set(xlabel='distance modulus μ', ylabel=r'$2\Delta\ln\mathcal{L}$ (profiled over f)', ylim=(0, 60))
ax[0].legend()
# right: the satellite density s(c, m | mu_hat) on the grid (reshaped to (90, 130)) and the stars that were fitted
ax[1].imshow(sat_density(GC, GM, mu_hat).reshape(GC.shape).T, extent=[gc[0], gc[-1], gm[-1], gm[0]], aspect='auto', cmap='Greys')
ax[1].scatter(c0[sel], g0[sel], s=4, c='C3')
ax[1].set(xlabel='$(g-r)_0$', ylabel='$g_0$', title=f'model s(c, m | μ = {mu_hat:.2f}) and stars inside 1.5 $r_h$')
plt.tight_layout()
# Test (STRUCTURE.md): mu within 0.3 mag of LVDB (19.11). The tolerance is much wider than the statistical error
# (+-0.02) because the systematic error dominates: isochrone age and [Fe/H], zero points, extinction (see text).
# Measured: 18.99, 0.12 mag off.
assert abs(mu_hat - ref.distance_modulus) < 0.3

**Figure 2.2.** Left: the profile likelihood $2\Delta\ln\mathcal L$ of the distance modulus μ (blue; maximised over
the satellite fraction f at each μ; 0.1-mag steps over 17.5–21.0, refined to 0.01 mag within ±0.3 mag of the coarse
minimum), for the stars inside 1.5 $r_h$ with $-0.4 < (g-r)_0 < 1.4$ and $16 < g_0 < 22.5$. The dotted line marks
$2\Delta\ln\mathcal L = 1$ (the 68% interval), the red dashed line the LVDB value μ = 19.11 ± 0.08. The clear minimum,
$\hat\mu$ = 18.99, is 0.12 mag from the published value; its statistical width (68% interval 18.97–19.01, i.e.
±0.02) is much smaller than that offset. Right: the satellite model $s(c, m \mid \hat\mu)$ at the best distance
(grey: the IMF-weighted isochrone blurred by the photometric errors plus 0.03 mag in colour and 0.1 mag in magnitude,
darker where more stars are expected) with the stars of the on-region used in the fit (red), most of which are field
stars.

The statistical error is therefore only a small part of the uncertainty: the result depends on the assumed age and metallicity
(an older or more metal-poor isochrone is fainter at fixed colour), on the photometric zero points, and on the
extinction. Try `logage = 10.0` or `feh = -1.5` above and see how far $\hat\mu$ moves.

## Summary

- On − off Hess diagrams reveal a satellite's CMD sequence; their errors are Poisson and include the scaled background.
- An isochrone shifted by the distance modulus, weighted by the IMF and blurred by the errors, is a CMD probability density.
- A mixture likelihood, profiled over the satellite fraction, measures μ for Boötes I within 0.3 mag of the literature.